Loading in Libraries


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
import numpy as np
import subprocess
import pandas as pd
import numpy as np
from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from IPython.display import Image
#import pydotplus as pydot
from sklearn import tree
warnings.filterwarnings('ignore')
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# **Date Cleaning and Loading**

**cleaning and doing one hot encoding**

In [ ]:
url = 'https://raw.githubusercontent.com/finsaccount/Intrusion-Detection-System-Machine-Learning/refs/heads/main/data%202.csv'
data = pd.read_csv(url)
labels_dict={'attack':1, 'benign':0}
data['label2'] = data['label2'].map(labels_dict)
data = data.dropna()
data = data.drop('label', axis=1)
data.head()


In [ ]:
X = data.drop('label2', axis=1)
Y = data.pop('label2')
x_train, x_test, y_train, y_test = train_test_split(X, Y,
                                                    test_size=0.3,
                                                    random_state=123,
                                                    shuffle=True)

## Model's

**Support Vector Machine**

In [ ]:
from sklearn.svm import SVC
svm_model = SVC(kernel='linear', C=100, gamma = 1)
svm_model.fit(x_train, y_train)

**KNN**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train.values, y_train.values)

**Random Forest**

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)
predictionTree = rf_model.predict(x_test)

# Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

svm_model = SVC(kernel='linear', C=100, gamma=1, probability=True)
knn_model = KNeighborsClassifier(n_neighbors=5)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
voting_clf = VotingClassifier(
    estimators=[('svm', svm_model), ('knn', knn_model), ('rf', rf_model)],
    voting='soft'
)
voting_clf.fit(x_train, y_train)
predictions = voting_clf.predict(x_test)

In [ ]:
print(classification_report(y_test, predictions))

In [ ]:
svm_model.fit(x_train, y_train)
predictionsSVM = svm_model.predict(x_test)
cm = confusion_matrix(y_test, predictionsSVM, labels=y_test.unique())


In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
print(classification_report(y_test,predictionsSVM))

In [ ]:
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.tree import DecisionTreeClassifier, export_graphviz
import pydot
from IPython.display import Image


rf_model.fit(x_train, y_train)
individual_tree = rf_model.estimators_[0]

export_graphviz(individual_tree, out_file='raw.dot',
                feature_names=list(x_train),
                class_names=label_names,
                rounded=True, filled=True)
(graph,) = pydot.graph_from_dot_file('raw.dot')
graph.write_png('individual_tree.png')

In [ ]:
predictionsRF = rf_model.predict(x_test)
print(classification_report(y_test, predictionsRF))
cm = confusion_matrix(y_test, predictionsRF, labels=y_test.unique())
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot()

In [ ]:
print(classification_report(y_test, predictions))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

#  'attack': 1, 'benign': 0
label_names = [name for name, val in sorted(labels_dict.items(), key=lambda item: item[1])]

cm = confusion_matrix(y_test, predictions, labels=y_test.unique())
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot()

# Stacking Classifier

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

estimators = [
    ('svm', SVC(probability=True)),
    ('knn',KNeighborsClassifier()),
    ('rf', RandomForestClassifier())
]
stack_clf = StackingClassifier(
    estimators = estimators,
    final_estimator = LogisticRegression()
)
stack_clf.fit(x_train,y_train)

In [ ]:
predictionsStack = stack_clf.predict(x_test)
print(classification_report(y_test, predictionsStack))
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

#  'attack': 1, 'benign': 0
label_names = [name for name, val in sorted(labels_dict.items(), key=lambda item: item[1])]

cm = confusion_matrix(y_test, predictionsStack, labels=y_test.unique())
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot()

# Bagging Classifier

In [ ]:
from sklearn.ensemble import BaggingClassifier
bag_dt = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50)
bag_dt.fit(x_train,y_train)
print(classification_report(y_test, bag_dt.predict(x_test)))

In [ ]:
bag_knn = BaggingClassifier(estimator=KNeighborsClassifier(n_neighbors=5), n_estimators=50)
bag_knn.fit(x_train, y_train)
print(classification_report(y_test, bag_knn.predict(x_test)))

In [ ]:
bag_svm = BaggingClassifier(estimator=SVC(), n_estimators=10)
bag_svm.fit(x_train, y_train)
print(classification_report(y_test, bag_knn.predict(x_test)))

# Summary

The Stacking Classifier worked best.
It produced a model with an accuracy of 98%